# 환경설정

In [ ]:
# 인증

from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

# stg_votes__vote — 투표(PING) 기록 스테이징

**목적**: `raw_userquestionrecord`(투표 기록)를 분석용 Stage로 정제
**산출물**: `votes.stg_votes__vote` (VIEW)
**Grain**: 1행 = 1건의 PING (발신자가 특정 질문에서 수신자를 선택한 기록)

**핵심 결정 요약**
- 시각: 전부 UTC → KST(`Asia/Seoul`) 변환
- 힌트 열람 정의: `status='I'` ❌ → **`opened_times > 0`** ✅ (열고 차단한 케이스까지 포착)
- `answered_at`: 실제 답장(P/A)일 때만 유지, 미답변(N)은 NULL (원본은 미답변도 시각이 채워져 오염됨)
- ⚠️ 주체 주의: 힌트·답장 필드는 전부 **수신자(`receiver_user_id`)의 행동**

**Raw 품질**: 중복 0 · 결측 0 · status 예상 외 값 0 · PK(id) 유니크

## Raw 품질 확인

### 1. 컬럼 타입 확인 (타임존)
- created_at / answer_updated_at 이 TIMESTAMP인지 DATETIME인지 → TIMESTAMP 면 변환 필요

In [ ]:
%%bigquery df_types --project your-gcp-project

SELECT column_name, data_type
FROM `your-gcp-project.raw.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'raw_userquestionrecord'
ORDER BY ordinal_position

In [ ]:
df_types

- created_at, answer_updated_at = **TIMESTAMP** (UTC 저장)
- KST로 변환 필요 : `DATETIME(col, 'Asia/Seoul')` 사용

### 2. 타임존 변환 검증
- UTC→KST 변환한 시각이 이용자 특성 답게 방과후~밤에 집중되어 있는지 눈으로 확인
- 새벽에 몰리면 타임존이 틀린 것
- 밤 시간대 집중, 새벽 3~5시 최저 → 타임존 변환 정상

In [ ]:
%%bigquery df_hour --project your-gcp-project
SELECT
  EXTRACT(HOUR FROM DATETIME(created_at, 'Asia/Seoul')) AS hour_kst,
  COUNT(*) AS cnt
FROM `your-gcp-project.raw.raw_userquestionrecord`
GROUP BY hour_kst
ORDER BY hour_kst

In [ ]:
df_hour

In [ ]:
!pip install koreanize-matplotlib
import koreanize_matplotlib
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.bar(df_hour['hour_kst'], df_hour['cnt'])
plt.xlabel('Hour (KST)')
plt.ylabel('Count')
plt.title('시간대별 투표 수')
plt.xticks(range(0, 24))
plt.show()

### 3. 중복 확인
- 중복, 결측값 개수 모두 0

In [ ]:
%%bigquery df_vote_pk_check --project your-gcp-project

SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT id) AS distinct_vote_id_count,
  COUNT(*) - COUNT(DISTINCT id) AS duplicate_vote_id_count,
  COUNTIF(id IS NULL) AS null_vote_id_count
FROM `your-gcp-project.raw.raw_userquestionrecord`;

In [ ]:
df_vote_pk_check

### 4. 결측 확인

In [ ]:
%%bigquery df_vote_null_check --project your-gcp-project

SELECT
  COUNT(*) AS row_count,

  COUNTIF(id IS NULL) AS null_vote_id_count,
  COUNTIF(user_id IS NULL) AS null_sender_user_id_count,
  COUNTIF(chosen_user_id IS NULL) AS null_receiver_user_id_count,
  COUNTIF(question_id IS NULL) AS null_question_id_count,
  COUNTIF(question_piece_id IS NULL) AS null_question_piece_id_count,
  COUNTIF(created_at IS NULL) AS null_created_at_count,

  COUNTIF(opened_times IS NULL) AS null_opened_times_count,
  COUNTIF(opened_times < 0) AS negative_opened_times_count
FROM `your-gcp-project.raw.raw_userquestionrecord`;

In [ ]:
df_vote_null_check

## 5. status 코드 분포
- 명세(C/I/B) 외 값이 있는지 체크
- C(닫힘) 115.6만 / I(초성열림) 6만 / B(차단) 658, 예상 외 값 0 → 디코딩 CASE 확정

In [ ]:
%%bigquery df_status --project your-gcp-project
SELECT
  status AS status_code,
  COUNT(*) AS cnt,
  COUNTIF(status NOT IN ('C','I','B')) AS unexpected  -- 0이어야 정상
FROM `your-gcp-project.raw.raw_userquestionrecord`
GROUP BY status
ORDER BY cnt DESC

In [ ]:
df_status

## 6. opened_times 확인
- status별 분포로 실제 열람 횟수를 의미하는 것이 맞는지 확인
- C (닫힘) : 전부 0 / I (초성열림) : 1·2·3 분포 / B (차단) : 0~3 (열고 차단 84건)
- → opened_times는 **진짜 "열람 횟수"** = 몰입 강도 지표로 활용 가능
- → 힌트 열람 정의를 `status='I'` 대신 **`opened_times > 0`** 으로 확정
  (B 중 보고 차단 케이스까지 포착되어 더 정확)

In [ ]:
%%bigquery df_opened --project your-gcp-project
SELECT
  status AS status_code,
  opened_times,
  COUNT(*) AS cnt
FROM `your-gcp-project.raw.raw_userquestionrecord`
GROUP BY status, opened_times
ORDER BY status, opened_times

In [ ]:
df_opened

## 7. 답장 시간 정합성 확인
- `answer_updated_at`이 미답변(N)에도 값이 있는지 확인 → 있으면 "답장 시각"으로 그대로 쓰면 안 됨
- **결과**: N(미답변) 109.7만 건도 전부 값 보유 (null 0건)
- **결정**: `answered_at`은 `answer_status IN ('P','A')`일 때만 유지, 미답변이면 NULL
  → "답장까지 걸린 시간" 계산 시 미답변이 0으로 섞이는 오염 방지

In [ ]:
%%bigquery df_vote_answer_time_check --project your-gcp-project



SELECT

  answer_status,

  COUNT(*) AS row_count,

  COUNTIF(answer_updated_at IS NULL) AS null_answered_at_count,

  COUNTIF(answer_updated_at IS NOT NULL) AS not_null_answered_at_count

FROM `your-gcp-project.raw.raw_userquestionrecord`
GROUP BY answer_status

ORDER BY row_count DESC;


In [ ]:
df_vote_answer_time_check

In [ ]:
%%bigquery df_vote_answer_time_check_1 --project your-gcp-project

SELECT
    answer_status,
    COUNT(*) AS cnt,
    COUNTIF(answer_updated_at = created_at) AS same_time,
    COUNTIF(answer_updated_at > created_at) AS updated_later
FROM `your-gcp-project.raw.raw_userquestionrecord`
GROUP BY answer_status;


In [ ]:
df_vote_answer_time_check_1

## 8. stg_votes__vote 최종 정제 미리보기
- BigQuery에서 쿼리 실행하고, 그 결과를 df_vote라는 Pandas DataFrame으로 가져와라
- (stage vote로 만들고 싶은 결과를 미리 Python에서 확인한 것)

In [ ]:
%%bigquery df_vote --project your-gcp-project

WITH src AS (
  SELECT
    id,
    user_id,
    chosen_user_id,
    question_id,
    question_piece_id,
    status,
    opened_times,
    answer_status,
    answer_updated_at,
    has_read,
    created_at
  FROM `your-gcp-project.raw.raw_userquestionrecord`
)

SELECT
  id AS vote_id,
  user_id AS sender_user_id,
  chosen_user_id AS receiver_user_id,

  question_id,
  question_piece_id,

  status AS status_code,
  CASE status
    WHEN 'C' THEN '닫힘'
    WHEN 'I' THEN '초성열림'
    WHEN 'B' THEN '차단'
    ELSE '알수없음'
  END AS status_label,

  opened_times,

  -- 힌트 열람 여부: status가 아니라 opened_times 기준
  COALESCE(opened_times > 0, FALSE) AS is_hint_opened,

  answer_status AS answer_status_code,
  CASE answer_status
    WHEN 'N' THEN '미답변'
    WHEN 'P' THEN '비공개'
    WHEN 'A' THEN '공개'
    ELSE '알수없음'
  END AS answer_status_label,

  COALESCE(answer_status IN ('P', 'A'), FALSE) AS is_answered,

  has_read,

  -- 투표 발생 시각: KST 변환
  DATETIME(created_at, 'Asia/Seoul') AS voted_at,

  -- 원본 answer_updated_at은 미답변에도 값이 있으므로 상태 변경 시각으로 보존
  DATETIME(answer_updated_at, 'Asia/Seoul') AS answer_status_updated_at,

  -- 실제 답장 시각: P/A일 때만 인정
  CASE
    WHEN answer_status IN ('P', 'A')
      THEN DATETIME(answer_updated_at, 'Asia/Seoul')
  END AS answer_updated_at

FROM src
ORDER BY created_at DESC
LIMIT 30;

In [ ]:
df_vote.head(5)

## 최종 Stage

### view (관례)

In [ ]:
from google.cloud import bigquery
from google.api_core.exceptions import Conflict

PROJECT_ID = "your-gcp-project"
STAGE_DATASET = "stage"
client = bigquery.Client(project=PROJECT_ID)

dataset_id = f"{PROJECT_ID}.{STAGE_DATASET}"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "asia-northeast3"

try:
    client.create_dataset(dataset)
    print(f"데이터셋 생성 완료: {dataset_id}")
except Conflict:
    print(f"이미 존재함: {dataset_id}")

In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE VIEW `your-gcp-project.stage.stg_vote` AS
WITH src AS (
  SELECT
    id, user_id, chosen_user_id, question_id, question_piece_id,
    status, opened_times, answer_status, answer_updated_at,
    has_read, created_at
  FROM `your-gcp-project.raw.raw_userquestionrecord`
)
SELECT
  id             AS vote_id,
  user_id        AS sender_user_id,
  chosen_user_id AS receiver_user_id,
  question_id,
  question_piece_id,
  status AS status_code,
  CASE status WHEN 'C' THEN '닫힘' WHEN 'I' THEN '초성열림' WHEN 'B' THEN '차단' END AS status_label,
  opened_times,
  COALESCE(opened_times > 0, FALSE) AS is_hint_opened,
  answer_status AS answer_status_code,
  CASE answer_status WHEN 'N' THEN '미답변' WHEN 'P' THEN '비공개' WHEN 'A' THEN '공개' END AS answer_status_label,
  COALESCE(answer_status IN ('P','A'), FALSE) AS is_answered,
  has_read,
  DATETIME(created_at, 'Asia/Seoul')        AS voted_at,
  DATETIME(answer_updated_at, 'Asia/Seoul') AS answer_status_updated_at,
  CASE WHEN answer_status IN ('P','A')
       THEN DATETIME(answer_updated_at, 'Asia/Seoul')
  END AS answered_at
FROM src;

### 생성 후 검증

In [ ]:
%%bigquery --project your-gcp-project
SELECT
  COUNT(*)                                                   AS total_rows,          -- 1,217,558 이어야
  COUNTIF(is_answered = FALSE AND answered_at IS NOT NULL)   AS bug_answered_at,     -- 0 이어야 (오염 제거 확인)
  COUNTIF(is_answered)                                       AS answered_rows        -- 119,626 이어야 (P 7,865 + A 111,761)
FROM `your-gcp-project.raw.stg_votes__vote`;